# Cross-talk and demixing

Notebook 1 built a recording **forward** and showed that the movie is a sum of
rank-one layers, `movie = A·C + b·f`: each cell `i` contributes its spatial footprint
`A_i` times its calcium trace `C_i`, on top of a diffuse background `b·f`. This
notebook asks the question that motivates the whole analysis pipeline: **given the
movie, how do you read one cell's calcium back out?**

The obvious answer - draw a region of interest around the cell and average its pixels
- is wrong, and *understanding why* is the point. Optical blur and 1-photon tissue
scatter (Notebook 1's optics stage) spread every footprint, so a cell's ROI also
collects light from its **neighbours** (cross-talk) and sits on the **neuropil**
pedestal. The naive trace is a mixture, not the cell.

The fix is **demixing**: solve the linear system `movie = A·C + b·f` for `C` instead
of masking. Because minisim generated the recording, we hold the true `A`, `C`, and
`b` - so we can show the contamination exactly, then show demixing remove it, and
finally find the regime where even *perfect-knowledge* demixing fails.

Three parts:

1. **The problem** - the naive footprint-ROI trace, and an honest accounting of what
   contaminates it.
2. **The fix** - demixing as the inverse of `A·C`, the factorization CNMF performs.
3. **The limit** - when footprints overlap too much, even oracle demixing breaks down.

> Like Notebook 3, this notebook is **static**: each demo shows the whole effect, so it
> reads correctly without a live kernel. Every demo cell starts with a `# try:` knob.

## Setup

We need a recording with genuine cross-talk. The CI fixture `make_recording` places
cells on a well-separated grid *to avoid* overlap - the opposite of what we want here
- so we pack many cells into a small field of view: the grid spacing shrinks until the
blurred, scattered footprints overlap, which is exactly the crowded regime real
cortex presents. We append a `Neuropil` step so the background pedestal is in play
too, and keep the true `A`, `C`, and background fields from `ground_truth`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from minisim.metrics import footprint_mask, footprint_roi_trace, trace_pearson, activity_similarity
from minisim.spec import Neuropil
from minisim.testing import make_recording

%matplotlib inline
plt.rcParams["image.cmap"] = "magma"
plt.rcParams["figure.dpi"] = 100

# A crowded recording: 64 cells in a 96 um FOV at 100 um depth, with neuropil. At this
# density the grid spacing (~8 um) is comparable to a blurred footprint, so footprints
# overlap and the naive ROI traces are contaminated. `detectable_subset()` is the fair
# population to study (Notebook 1's detection floor; Notebook 3's honest denominator).
rec = make_recording(n_cells=64, n_px=96, pixel_size_um=1.0, depth_um=100.0,
                     duration_s=24.0, seed=0, extra_steps=[Neuropil()])
gt = rec.ground_truth
movie = np.asarray(rec.observed, float)
A = np.asarray(gt.A_observed)               # (n, H, W) degraded footprints - the spatial model
C = np.asarray(gt.C)                         # (n, frames) true calcium - the answer key
centers = np.asarray(gt.centers_um)          # (n, 3) (z, y, x) soma centers
fps = rec.spec.acquisition.fps
t = np.arange(movie.shape[0]) / fps
print(f"{gt.n_units} cells, {int(np.asarray(gt.detectable).sum())} detectable; "
      f"movie {movie.shape}")


def corr(a, b):
    # Pearson correlation between two 1-D traces (0 when either is flat).
    a = np.asarray(a, float) - np.mean(a); b = np.asarray(b, float) - np.mean(b)
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / d) if d else 0.0


def nearest_others(i, k=3):
    # Indices of the k nearest OTHER cells, by lateral (y, x) distance.
    d = np.linalg.norm(centers[:, 1:] - centers[i, 1:], axis=1)
    return [j for j in np.argsort(d) if j != i][:k]

# Part 1 - the problem: cross-talk

A footprint ROI is what a person draws by eye: the bright pixels of one cell. Averaging
the movie over that mask (`footprint_roi_trace`) is the simplest possible read-out.
This part shows that read-out is a *mixture* and accounts for what is in it.

## 1.1 The naive read-out does not recover the cell

`footprint_roi_trace` averages the movie over a cell's footprint mask - exactly a
hand-drawn ROI, with no unmixing. Compared against the true `C` for a crowded cell, it
picks up transients the cell never fired: those are the **neighbours'** events,
blurred and scattered into the mask. (Traces are z-scored so the *shape* mismatch is
what shows - the naive trace is also in arbitrary fluorescence units.)

In [ ]:
def z(x):
    x = np.asarray(x, float); return (x - x.mean()) / (x.std() or 1.0)


def most_contaminated(n=3):
    # Detectable cells whose naive ROI correlates most with a NEIGHBOUR's true C -
    # the clearest victims of cross-talk (typically quiet cells beside busy ones).
    det = np.where(np.asarray(gt.detectable))[0]
    score = [max(corr(footprint_roi_trace(movie, A[i]), C[j]) for j in nearest_others(i))
             for i in det]
    return det[np.argsort(score)[-n:]]


def show_naive_vs_true():
    # try: pick your own cells, e.g. picks = [5, 12, 30].
    picks = most_contaminated(3)
    fig, axes = plt.subplots(len(picks), 1, figsize=(9, 5.5), sharex=True)
    for ax, i in zip(np.atleast_1d(axes), picks):
        naive = footprint_roi_trace(movie, A[i])
        ax.plot(t, z(C[i]), color="k", lw=1.3, label="true C")
        ax.plot(t, z(naive), color="C1", lw=0.9, label="naive ROI")
        ax.set_ylabel(f"cell {i}"); ax.set_yticks([])
        ax.legend(fontsize=7, ncol=2, loc="upper right")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    np.atleast_1d(axes)[0].set_title("naive footprint-ROI trace vs the true calcium C")
    plt.show()


show_naive_vs_true()

## 1.2 What is in the naive trace?

Because we hold every cell's true `C` and the background driver, we can *attribute* the
naive ROI to its sources. Regress the naive trace onto the cell's own `C`, its nearest
neighbours' `C`, and the neuropil population driver `P(t)`: the fitted pieces add back
up to the naive trace, and each piece is one contamination term:

`naive ROI  ≈  (own C)  +  (neighbour bleed)  +  (neuropil pedestal)  +  offset`.

The own-cell piece is the signal we wanted; everything else is contamination the ROI
cannot separate.

In [ ]:
def attribute_roi(i, k=3):
    # Least-squares attribution of the naive ROI to true sources: own C, k nearest
    # neighbours' C, the neuropil driver P(t), and a constant. Returns each fitted
    # contribution (these sum to the model's fit of the naive trace).
    naive = footprint_roi_trace(movie, A[i])
    nbrs = nearest_others(i, k)
    P = (np.asarray(gt.neuropil_population, float) if gt.neuropil_population is not None
         else np.ones(movie.shape[0]))
    cols = [C[i]] + [C[j] for j in nbrs] + [P, np.ones_like(P)]
    X = np.array(cols, float).T                       # (frames, 2 + k)
    beta, *_ = np.linalg.lstsq(X, naive, rcond=None)
    contrib = X * beta                                 # per-source fitted contribution
    own = contrib[:, 0]
    neigh = contrib[:, 1:1 + k].sum(axis=1)
    pedestal = contrib[:, 1 + k]
    return naive, own, neigh, pedestal, nbrs


def show_contamination_anatomy():
    # try: change the cell, or k (number of neighbours attributed).
    i = most_contaminated(1)[0]
    naive, own, neigh, pedestal, nbrs = attribute_roi(i)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t, naive, color="k", lw=1.4, label="naive ROI (measured)")
    ax.plot(t, own, color="C0", lw=1.0, label="own C (the signal)")
    ax.plot(t, neigh, color="C3", lw=1.0, label=f"neighbour bleed ({len(nbrs)} nearest)")
    ax.plot(t, pedestal - pedestal.mean(), color="C2", lw=1.0, label="neuropil pedestal")
    ax.set(xlabel="time (s)", ylabel="ROI fluorescence (a.u.)",
           title=f"cell {i}: the naive ROI decomposed into its sources")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    plt.show()


show_contamination_anatomy()

Across the whole detectable population, the symptom is systematic: plot each cell's
naive-ROI correlation with its **own** `C` against its correlation with its **nearest
neighbour's** `C`. A clean read-out would sit at the bottom-right (high own, zero
neighbour). Instead the quiet cells - little signal of their own to dominate the mask -
drift up and left, their traces driven by whoever is firing next door.

In [ ]:
def show_crosstalk_scatter():
    # try: colour by depth (centers[det, 0]) instead of activity.
    det = np.where(np.asarray(gt.detectable))[0]
    own = np.array([corr(footprint_roi_trace(movie, A[i]), C[i]) for i in det])
    nbr = np.array([max(corr(footprint_roi_trace(movie, A[i]), C[j])
                        for j in nearest_others(i)) for i in det])
    activity = C[det].std(axis=1)                    # how much signal the cell has itself
    fig, ax = plt.subplots(figsize=(5.6, 5))
    sc = ax.scatter(own, nbr, c=activity, cmap="viridis", s=26)
    ax.plot([0, 1], [0, 1], ls="--", color="0.7", lw=1)
    ax.set(xlabel="corr(naive ROI, own C)", ylabel="corr(naive ROI, nearest neighbour C)",
           title="cross-talk: quiet cells track their neighbours", xlim=(0, 1.02), ylim=(-0.1, 1))
    fig.colorbar(sc, ax=ax, label="own activity (std of C)")
    plt.show()


show_crosstalk_scatter()

## 1.3 Why: the footprints physically overlap

Cross-talk is not a measurement mistake - it is in the data. Notebook 1's optics stage
blurred each footprint (diffraction + defocus) and the tissue scattered it; at this
depth and density the resulting footprints overlap, so one cell's ROI mask sits on top
of its neighbours' light. The sum of all footprints shows the crowding directly, and a
cell's mask (outlined) almost always overlaps a neighbour's.

In [ ]:
def show_footprint_overlap():
    # try: outline a different cell's ROI mask.
    i = most_contaminated(1)[0]
    total = A.sum(axis=0)
    mask = footprint_mask(A[i])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 4.6))
    axl.imshow(total); axl.set_title(f"all {gt.n_units} footprints summed (light is shared)")
    axl.contour(mask, levels=[0.5], colors="w", linewidths=1.2)
    axl.set_xticks([]); axl.set_yticks([])
    # Zoom on the picked cell to see whose light its mask collects.
    ys, xs = np.where(mask); pad = 14
    y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, total.shape[0])
    x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, total.shape[1])
    axr.imshow(total[y0:y1, x0:x1])
    axr.contour(mask[y0:y1, x0:x1], levels=[0.5], colors="w", linewidths=1.4)
    axr.set_title(f"cell {i}'s ROI mask sits on neighbours' footprints")
    axr.set_xticks([]); axr.set_yticks([])
    plt.show()


show_footprint_overlap()

# Part 2 - the fix: demixing

The naive ROI fails because masking ignores the spatial model. Demixing uses it: the
movie *is* `A·C + b·f`, a linear system, so given the spatial maps we can solve for the
traces instead of averaging pixels.

## 2.1 Solve the system instead of masking

Stack every known spatial map into one matrix `D` - one column per cell footprint
`A_i`, one per neuropil field `b_k`, and a constant for any flat offset (leakage). Each
frame of the movie is then `movie[t] = D · x[t]`, and least squares recovers `x[t]`:
the first rows are the cells' demixed traces, the rest the background.

This is an **oracle** demix - it uses the *true* footprints, the most any method could
know. A real pipeline (CNMF) must also *discover* `A`; here we isolate the demixing
step alone, to see what it can and cannot do even with perfect spatial knowledge.

In [ ]:
def oracle_demix(movie, A, neuropil_spatial):
    # Solve movie[t] = D @ x[t] for all frames at once. D's columns are the known
    # footprints, the known neuropil fields, and a constant; lstsq returns the traces.
    T, H, W = movie.shape
    n = A.shape[0]
    cols = [A.reshape(n, -1)]
    if neuropil_spatial is not None:
        cols.append(np.asarray(neuropil_spatial).reshape(-1, H * W))
    cols.append(np.ones((1, H * W)))                 # constant (leakage / offset)
    D = np.concatenate(cols, axis=0).T               # (HW, n + K + 1)
    X, *_ = np.linalg.lstsq(D, movie.reshape(T, -1).T, rcond=None)
    return X[:n]                                     # (n, frames) demixed cell traces


C_demix = oracle_demix(movie, A, gt.neuropil_spatial)


def show_demix_vs_naive():
    # try: pick your own cells.
    picks = most_contaminated(3)
    fig, axes = plt.subplots(len(picks), 1, figsize=(9, 5.5), sharex=True)
    for ax, i in zip(np.atleast_1d(axes), picks):
        ax.plot(t, z(footprint_roi_trace(movie, A[i])), color="C1", lw=0.8, label="naive ROI")
        ax.plot(t, z(C_demix[i]), color="C2", lw=1.0, label="demixed")
        ax.plot(t, z(C[i]), color="k", lw=1.0, ls="--", label="true C")
        ax.set_ylabel(f"cell {i}"); ax.set_yticks([])
        ax.legend(fontsize=7, ncol=3, loc="upper right")
    np.atleast_1d(axes)[-1].set_xlabel("time (s)")
    np.atleast_1d(axes)[0].set_title("demixing recovers C where the naive ROI could not")
    plt.show()


show_demix_vs_naive()

## 2.2 Score it: contamination gone, traces recovered

The fix is not just visual. For every detectable cell, compare the naive ROI and the
demixed trace on two scores from Notebook 3: the trace correlation with the true `C`
(higher is better), and the correlation with the nearest neighbour's `C` (the
cross-talk - lower is better). Demixing pushes own-correlation up and neighbour-
correlation to ~0.

In [ ]:
def show_recovery_scores():
    det = np.where(np.asarray(gt.detectable))[0]
    pairing = [(i, i) for i in det]
    naive_stack = np.array([footprint_roi_trace(movie, A[i]) for i in det])
    own_naive = trace_pearson(naive_stack, C[det], [(k, k) for k in range(len(det))])
    own_demix = trace_pearson(C_demix[det], C[det], [(k, k) for k in range(len(det))])
    nbr_naive = np.array([max(corr(naive_stack[k], C[j]) for j in nearest_others(i))
                          for k, i in enumerate(det)])
    nbr_demix = np.array([max(corr(C_demix[i], C[j]) for j in nearest_others(i)) for i in det])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, naive_v, demix_v, ttl in [
        (axl, own_naive, own_demix, "correlation with own C (higher = better)"),
        (axr, nbr_naive, nbr_demix, "correlation with neighbour C (lower = better)"),
    ]:
        ax.hist(naive_v, bins=np.linspace(-0.2, 1, 25), alpha=0.6, color="C1", label="naive ROI")
        ax.hist(demix_v, bins=np.linspace(-0.2, 1, 25), alpha=0.6, color="C2", label="demixed")
        ax.set(title=ttl, xlabel="correlation", ylabel="cells"); ax.legend(fontsize=8)
    plt.show()
    print(f"median own-C corr   naive {np.nanmedian(own_naive):.2f}  ->  demix {np.nanmedian(own_demix):.2f}")
    print(f"median neighbour corr naive {np.median(nbr_naive):.2f}  ->  demix {np.median(nbr_demix):.2f}")


show_recovery_scores()

## 2.3 This is what CNMF does

The solve above *is* the `A·C` factorization run backward. CNMF (minian's core)
estimates the same `A` and `C` from the movie - the difference is that it has no oracle:
it must **discover** the footprints `A` and the background `b` at the same time as the
traces, alternating between updating the spatial maps and the temporal ones. Our oracle
demix is the best case that bounds it: it shows that *once the spatial model is right*,
the cross-talk is removable. The pipeline's job is to get the spatial model right, and
Notebook 3's metrics score how well it did.

So the forward chain of Notebook 1 and the inverse here meet: `A·C + b·f` built up from
physics, and `A·C + b·f` taken apart to read the biology back out.

# Part 3 - the limit: when even oracle demixing fails

Demixing is a linear solve, and a linear solve has a condition number. When footprints
overlap only a little, their columns in `D` are nearly independent and the solve is
easy. As cells crowd closer (higher density, or deeper cells with more scatter), the
columns become nearly parallel - two footprints the optics can no longer tell apart -
and the system becomes ill-conditioned. Past that point, no demixer, however perfect
its knowledge of `A`, can separate the traces: the information is gone from the movie.

Sweep density (cells in the same FOV) and watch both read-outs. The naive ROI degrades
throughout; oracle demixing holds near-perfect until the footprints overlap too much,
then it too collapses - the irreducible limit any pipeline inherits.

In [ ]:
def show_density_limit():
    # try: change the densities, or sweep depth_um instead of n_cells. One short
    # recording per point (kept brief so the sweep stays quick).
    counts = [16, 36, 64, 100, 144]
    naive_r, demix_r, n_det = [], [], []
    for n in counts:
        r = make_recording(n_cells=n, n_px=96, pixel_size_um=1.0, depth_um=120.0,
                           duration_s=12.0, seed=0, extra_steps=[Neuropil()])
        g = r.ground_truth; mv = np.asarray(r.observed, float)
        Aa = np.asarray(g.A_observed); Cc = np.asarray(g.C)
        det = np.where(np.asarray(g.detectable))[0]
        Cd = oracle_demix(mv, Aa, g.neuropil_spatial)
        pr = [(k, k) for k in range(len(det))]
        naive_stack = np.array([footprint_roi_trace(mv, Aa[i]) for i in det])
        naive_r.append(float(np.nanmedian(trace_pearson(naive_stack, Cc[det], pr))))
        demix_r.append(float(np.nanmedian(trace_pearson(Cd[det], Cc[det], pr))))
        n_det.append(len(det))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(counts, naive_r, marker="o", color="C1", label="naive ROI")
    ax.plot(counts, demix_r, marker="s", color="C2", label="oracle demix")
    ax.set(xlabel="cells in the 96 um FOV (density)", ylabel="median corr with true C",
           title="oracle demixing holds, then collapses as footprints overlap", ylim=(0, 1.02))
    for x, y, nd in zip(counts, demix_r, n_det):
        ax.annotate(f"{nd} det", (x, y), textcoords="offset points", xytext=(0, 8), fontsize=7)
    ax.legend(); ax.grid(alpha=0.3)
    plt.show()


show_density_limit()

## Recap

- A recording is `A·C + b·f`. Reading a cell out by **averaging an ROI** does not
  recover its calcium: optical blur and tissue scatter spread every footprint, so the
  mask also collects **neighbour bleed** and the **neuropil pedestal** (Part 1).
- **Demixing** - solving `movie = A·C + b·f` for the traces instead of masking -
  removes the cross-talk, recovering `C` where the naive ROI could not (Part 2). This is
  the `A·C` factorization **CNMF** performs, run as the inverse of Notebook 1's forward
  build; the difference is that CNMF must also *discover* `A`.
- Demixing has an **irreducible limit**: when footprints overlap too much, the linear
  system is ill-conditioned and even oracle demixing - perfect knowledge of `A` -
  cannot separate the traces (Part 3). That limit is physics, set by the optics and the
  density, and it bounds every analysis pipeline.

With the three notebooks together: Notebook 1 builds the recording forward from physics,
this notebook inverts it to read the biology back out, and Notebook 3 scores how well a
real pipeline does the same.